In [39]:
import optuna
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.datasets import cifar100
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pandas as pd
import datetime
import io
import contextlib
import json



In [40]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))


Num GPUs Available: 1


In [41]:
NUM_CLASSES = 100
INPUT_SHAPE = (32, 32, 3)
MAX_EPOCHS = 50
VALIDATION_SPLIT = 0.2

In [42]:
(x_train, y_train), (x_test, y_test) = cifar100.load_data(label_mode='fine')

# Normalize pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

y_train_cat = to_categorical(y_train, NUM_CLASSES)
y_test_cat = to_categorical(y_test, NUM_CLASSES)

In [ ]:
# Global list to store results
trial_log = []


def objective(trial):
    K.clear_session()

    # 🎯 Hyperparameter suggestions
    lr = trial.suggest_categorical("lr", [1e-5, 1e-4, 1e-3])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop"])
    dropout_rate = trial.suggest_categorical("dropout", [0.2, 0.3, 0.4])

    # Filters
    filters_1 = trial.suggest_categorical("filters_1", [64, 128, 256])  # always included
    filters_2 = trial.suggest_categorical("filters_2", [0, 64, 128, 256])
    filters_3 = trial.suggest_categorical("filters_3", [0, 64, 128, 256])

    dense_units = trial.suggest_categorical("dense_units", [128, 256, 512])
    batch_size = trial.suggest_categorical("batch_size", [64, 128])

    optimizer = Adam(learning_rate=lr) if optimizer_name == "Adam" else RMSprop(learning_rate=lr)
    
    print(f"\nTrial {trial.number} Hyperparameters:")
    print(f"  Learning rate: {lr}")
    print(f"  Optimizer:     {optimizer_name}")
    print(f"  Dropout:       {dropout_rate}")
    print(f"  Filters:       [{filters_1}, {filters_2}, {filters_3}]")
    print(f"  Dense units:   {dense_units}")
    print(f"  Batch size:    {batch_size}")

    # 🧱 Build model
    model = Sequential()

    # Conv Block 1 (always included)
    model.add(Conv2D(filters_1, (3, 3), padding="same", activation='relu', input_shape=INPUT_SHAPE))
    model.add(BatchNormalization())
    model.add(Conv2D(filters_1, (3, 3), padding="same", activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Dropout(dropout_rate))

    # Conv Block 2 (optional)
    if filters_2 > 0:
        model.add(Conv2D(filters_2, (3, 3), padding="same", activation='relu'))
        model.add(BatchNormalization())
        model.add(Conv2D(filters_2, (3, 3), padding="same", activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(dropout_rate))

    # Conv Block 3 (optional)
    if filters_3 > 0:
        model.add(Conv2D(filters_3, (3, 3), padding="same", activation='relu'))
        model.add(BatchNormalization())
        model.add(Conv2D(filters_3, (3, 3), padding="same", activation='relu'))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(dropout_rate))

    # Dense block
    model.add(Flatten())
    model.add(Dense(dense_units, activation='relu'))
    model.add(Dropout(dropout_rate))
    model.add(Dense(NUM_CLASSES, activation='softmax'))

    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    # Callbacks
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=0)

    # Train the model
    history = model.fit(
        x_train, y_train_cat,
        validation_split=VALIDATION_SPLIT,
        epochs=MAX_EPOCHS,
        batch_size=batch_size,
        verbose=1,
        callbacks=[early_stopping, reduce_lr]
    )

    val_acc = history.history['val_accuracy'][-1]
    test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
    
    with io.StringIO() as buf, contextlib.redirect_stdout(buf):
        model.summary()
        model_summary_str = buf.getvalue()


    trial_log.append({
        'trial': trial.number,
        'lr': lr,
        'optimizer': optimizer_name,
        'dropout': dropout_rate,
        'filters_1': filters_1,
        'filters_2': filters_2,
        'filters_3': filters_3,
        'dense_units': dense_units,
        'batch_size': batch_size,
        'val_accuracy': val_acc,
        'test_accuracy': test_acc,
        'epochs_ran': len(history.history['accuracy']),
        'timestamp': datetime.datetime.now().isoformat(),
        'model_summary': model_summary_str 
    })

    return val_acc


In [ ]:
try:
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=50, show_progress_bar=True)

except KeyboardInterrupt:
    print("\nOptimization interrupted. Saving progress...")

finally:
    # Save to CSV and JSON no matter what
    pd.DataFrame(trial_log).to_csv("optuna_trials.csv", index=False)

    with open("optuna_trials_with_summaries.json", "w") as f:
        json.dump(trial_log, f, indent=2)

    print("✅ Trials saved to 'optuna_trials.csv' and 'optuna_trials_with_summaries.json'")


[I 2025-03-27 00:19:27,035] A new study created in memory with name: no-name-1355ef1d-73fb-42d1-835b-706af4e32811
  0%|          | 0/50 [00:00<?, ?it/s]


Trial 0 Hyperparameters:
  Learning rate: 0.001
  Optimizer:     Adam
  Dropout:       0.26445933530085175
  Filters:       [256, 0, 64]
  Dense units:   256
  Batch size:    128


/Users/awabakram/git/cognitive_robotics_coursework/myenv/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 174s 551ms/step - accuracy: 0.0188 - loss: 15.5939 - val_accuracy: 0.0241 - val_loss: 13.5654 - learning_rate: 0.0010
Epoch 2/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 130s 415ms/step - accuracy: 0.0424 - loss: 25.3371 - val_accuracy: 0.0604 - val_loss: 12.2274 - learning_rate: 0.0010
Epoch 3/50
313/313 ━━━━━━━━━━━━━━━━━━━━ 128s 409ms/step - accuracy: 0.0622 - loss: 21.7521 - val_accuracy: 0.0889 - val_loss: 10.0527 - learning_rate: 0.0010
Epoch 4/50
291/313 ━━━━━━━━━━━━━━━━━━━━ 9s 409ms/step - accuracy: 0.0746 - loss: 13.1941